In [1]:
import torch 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import os

In [2]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/baselineModels/UNET'

In [3]:
from Unet_model import UNet

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
my_model = UNet()
my_model

UNet(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (4): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

In [6]:
my_model.to(device=device)

UNet(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (4): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

In [7]:
my_model.load_state_dict(torch.load("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/baselineModels/UNET/unet_rooftop_50_indian_usa.pth"))
my_model

UNet(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (4): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

In [8]:

root_dir = '/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images'
val_images_dir = os.path.join(root_dir, 'gandhinagar_dataset/test/images')
val_masks_dir = os.path.join(root_dir, 'gandhinagar_dataset/test/masks')

In [9]:
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import transforms
import cv2

In [10]:
# Custom Dataset
class RooftopDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_filenames = sorted(os.listdir(image_dir))
        self.mask_filenames = sorted(os.listdir(mask_dir))

        self.transform = transform
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_filenames[idx])
        
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = mask.astype(np.float32) / 255.0
        
        if self.transform:
            image = self.transform(image)
            mask = transforms.ToTensor()(mask).unsqueeze(0)  # Ensure mask shape [1, H, W]
        
        return image, mask.squeeze(0)  # Ensure mask shape [H, W]

# Transformations
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((1024, 1024)),
    transforms.ToTensor()
])

# Load datasets
val_dataset = RooftopDataset(val_images_dir, val_masks_dir, transform)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)


In [11]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/baselineModels/UNET'

In [12]:
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images")
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images'

In [13]:
from accuracy_indian import compute_metrics

In [14]:
# Evaluation

def evaluate(model, val_loader):
    model.eval()
    result_data = []
    # iou_scores = []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            # print(outputs.shape)
            preds = (outputs > 0.5).float()
            
            # one  = preds[0].squeeze(0).cpu().numpy()
            # plt.imshow(one, cmap="gray")
            # plt.tight_layout()
            # plt.show()
            
            # original = masks[0].squeeze(0).cpu().numpy()
            # plt.imshow(original, cmap="gray")
            # plt.tight_layout()
            # plt.show()

            metrics = compute_metrics(preds, masks)
            result_data.append(metrics)

    return result_data

In [15]:
accuracy_result = evaluate(my_model, val_loader)
accuracy_result

[[np.float64(0.8191891342356076),
  0.9006090887628536,
  0.9645934700965881,
  0.8780620616100678,
  0.9243445659438752,
  0.8191891342356076,
  0.9006090887628536,
  0.8780620616100678,
  0.9243445659438752,
  1.0],
 [np.float64(0.7791904633996546),
  0.875893255307571,
  0.9533047080039978,
  0.8379295712319882,
  0.917460196301305,
  0.7791904633996546,
  0.875893255307571,
  0.8379295712319882,
  0.917460196301305,
  1.0],
 [np.float64(0.7732786122280979),
  0.8721456480620211,
  0.9512774348258972,
  0.8401783537536287,
  0.9066417568266295,
  0.7732786122280979,
  0.8721456480620211,
  0.8401783537536287,
  0.9066417568266295,
  1.0],
 [np.float64(0.7751645095996672),
  0.8733438567611757,
  0.9657835593590369,
  0.8632840243833697,
  0.8836409075123418,
  0.7751645095996672,
  0.8733438567611757,
  0.8632840243833697,
  0.8836409075123418,
  1.0]]

In [16]:
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/baselineModels/UNET/unet_validation_results_usa.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

In [17]:
import pandas as pd 

metrics_df = pd.DataFrame(accuracy_result, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)

In [18]:
metrics_df.mean()

pixel_iou                  0.786706
pixel_dice                 0.880498
pixel_accuracy             0.958740
pixel_precision            0.854864
pixel_recall               0.908022
region_iou                 0.786706
region_dice                0.880498
region_precision           0.854864
region_recall              0.908022
region_success_accuracy    1.000000
dtype: float64

# UNET : From scratch 

pixel_iou                  0.772967
pixel_dice                 0.871760
pixel_accuracy             0.957248
pixel_precision            0.870724
pixel_recall               0.873367
region_iou                 0.772967
region_dice                0.871760
region_precision           0.870724
region_recall              0.873367
region_success_accuracy    1.000000


# UNET : From Pretrained 

pixel_iou                  0.786706
pixel_dice                 0.880498
pixel_accuracy             0.958740
pixel_precision            0.854864
pixel_recall               0.908022
region_iou                 0.786706
region_dice                0.880498
region_precision           0.854864
region_recall              0.908022
region_success_accuracy    1.000000
